# SLM-WM FlowHF one-prompt smoke

Select an A100-80GB runtime. Scientific work stays on `/content`; Drive is used only to read the request/key and deliver the final secret-scanned archive.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from google.colab import drive
repo = Path('/content/SLM-WM-FlowHF')
if repo.exists():
    raise RuntimeError('fresh Colab runtime required')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RICHAAARC/SLM-WM-FlowHF.git', str(repo)], check=True)
drive.mount('/content/drive', force_remount=False)
try:
    request_bytes = Path('/content/drive/MyDrive/SLM/flow-hf/inputs/run_request.json').read_bytes()
finally:
    drive.flush_and_unmount()
local_request = Path('/content/flowhf/input/run_request.json')
local_request.parent.mkdir(parents=True, exist_ok=True)
local_request.write_bytes(request_bytes)
expected_commit = json.loads(request_bytes)['repository_commit']
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'main'], cwd=repo, check=True)
remote_commit = subprocess.run(['git', 'ls-remote', 'origin', 'refs/heads/main'], cwd=repo, check=True, capture_output=True, text=True).stdout.split()[0]
clone_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
if expected_commit != remote_commit or clone_commit != remote_commit:
    raise RuntimeError('published main does not match the run request')
subprocess.run(['git', 'switch', '--detach', expected_commit], cwd=repo, check=True)
detached_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
if detached_commit != expected_commit:
    raise RuntimeError('detached checkout does not match the run request')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--upgrade', str(repo)], check=True)
from flowhf import colab_entry
drive_adapter = colab_entry.GoogleColabDriveAdapter()
prepared = colab_entry.load_verified_drive_input(adapter=drive_adapter, secret_getter=colab_entry.colab_secret_getter)


In [ ]:
from flowhf import colab_entry
colab_entry.run_prepared_input(prepared)
prepared = None


In [ ]:
# Run this cell independently after either success or failure.
import importlib
from flowhf import colab_entry
importlib.reload(colab_entry)
colab_entry.package_and_deliver_from_disk(adapter=colab_entry.GoogleColabDriveAdapter(), secret_getter=colab_entry.colab_secret_getter)
